# Data Cleaning - Complete Blood Count (CBC)

**Input:** `Dataset/Data/CBC.csv`

**Output:** `Dataset/Clean_Data/CBC_clean.csv`

---

## Vấn đề phát hiện từ EDA (I_CBC.ipynb)

| # | Vấn đề | Mô tả | Mức độ ảnh hưởng |
|---|--------|-------|------------------|
| 1 | Artifact kỹ thuật | Giá trị `5.397605346934028e-79` xuất hiện ở 4,284 ô, là placeholder hệ thống NHANES | Cao - gây nhiễu thống kê và mô hình |
| 2 | Missing values | 1,018 dòng thiếu hoàn toàn (11.1%) - không có kết quả xét nghiệm | Trung bình - cần xử lý trước khi modeling |
| 3 | Duplicates | Không có dòng trùng lặp hoàn toàn | Thấp |
| 4 | Biological outliers | 96.5% dòng có ≥1 giá trị ngoài range sinh lý tham khảo | Trung bình - cần review lâm sàng |
| 5 | Data type | Một số cột bị đọc nhầm kiểu do artifact | Trung bình - cần ép kiểu numeric |
| 6 | No target | Không có cột nhãn bệnh | Cao - cần merge với dataset khác |
1. **Artifact handling:** Thay thế toàn bộ `5.397605346934028e-79` bằng `NaN` đồng nhất.
2. **Numeric coercion:** Ép kiểu tất cả cột sang numeric với `errors='coerce'` để bắt giá trị không hợp lệ.
3. **Missing values:**
   - Xóa dòng thiếu hoàn toàn (all NaN) - 1,018 dòng (11.1%).
   - Cột `LBXMC` có tỷ lệ missing thấp hơn, giữ lại.
4. **Outliers:** Giữ nguyên giá trị, thêm cột flag `outlier_flag` đánh dấu các giá trị ngoài range sinh lý.
5. **Duplicates:** Kiểm tra và loại bỏ nếu có.
6. **Export:** Lưu kết quả ra `Clean_Data/CBC_clean.csv`.

---

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================
# CONFIGURATION
# ============================================
INPUT_PATH = "../Data/CBC.csv"
OUTPUT_DIR = "../Clean_Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "CBC_clean.csv")
ARTIFACT_VAL = 5.397605346934028e-79

# Biological reference ranges (adult)
BIOLOGICAL_RANGES = {
    "LBXWBCSI": (4.5, 11.0),
    "LBXLYPCT": (20.0, 40.0),
    "LBXMOPCT": (2.0, 8.0),
    "LBXNEPCT": (40.0, 60.0),
    "LBXEOPCT": (1.0, 4.0),
    "LBXBAPCT": (0.5, 1.0),
    "LBDLYMNO": (1.0, 3.0),
    "LBDMONO": (0.2, 0.8),
    "LBDNENO": (1.8, 7.0),
    "LBDEONO": (0.0, 0.5),
    "LBDBANO": (0.0, 0.2),
    "LBXRBCSI": (4.0, 5.5),
    "LBXHGB": (12.0, 17.5),
    "LBXHCT": (36.0, 50.0),
    "LBXMCVSI": (80.0, 100.0),
    "LBXMCHSI": (27.0, 31.0),
    "LBXMC": (32.0, 36.0),
    "LBXRDW": (11.5, 14.5),
    "LBXPLTSI": (150.0, 400.0),
    "LBXMPSI": (7.5, 11.5),
}

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(5)

In [ ]:
# Check for artifact values
artifact_before = (df == ARTIFACT_VAL).sum().sum()
print(f"Total artifact values before cleaning: {artifact_before}")

# Replace artifact with NaN
df_clean = df.replace(ARTIFACT_VAL, np.nan)

artifact_after = (df_clean == ARTIFACT_VAL).sum().sum()
print(f"Artifact values after cleaning: {artifact_after}")
print(f"Removed: {artifact_before - artifact_after}")

In [ ]:
# Ép kiểu tất cả cột (trừ ID) sang numeric với `errors='coerce'` để bắt các giá trị không hợp lệ.
# Identify ID column and numeric columns
id_col = "SEQN"
numeric_cols = [c for c in df_clean.columns if c != id_col]

# Convert to numeric, coerce errors to NaN
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("=== Data Types After Coercion ===")
print(df_clean.dtypes)

# Check for any new NaN introduced by coercion
new_nans = df_clean[numeric_cols].isnull().sum().sum() - df[numeric_cols].replace(ARTIFACT_VAL, np.nan).isnull().sum().sum()
print(f"\nNew NaN from coercion: {new_nans}")

In [ ]:
# Xử lý missing values: loại bỏ các hàng mà tất cả các cột numeric đều là NaN
rows_before = len(df_clean)

# Drop rows where ALL numeric columns are NaN
df_clean = df_clean.dropna(subset=numeric_cols, how="all")

rows_after = len(df_clean)
print(f"Rows before: {rows_before}")
print(f"Rows after removing all-NaN rows: {rows_after}")
print(f"Removed: {rows_before - rows_after} rows ({(rows_before-rows_after)/rows_before*100:.1f}%)")

print("\n=== Missing Values After Cleaning ===")
missing = df_clean[numeric_cols].isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
missing_df = pd.DataFrame({"Missing": missing, "Pct": missing_pct})
print(missing_df[missing_df["Missing"] > 0].sort_values("Pct", ascending=False))

In [ ]:
# Xử lý trùng lặp: loại bỏ các hàng trùng lặp
duplicates = df_clean.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows.")
else:
    print("No duplicates found.")

In [ ]:
# Xử lý giá trị vô lý: loại bỏ các hàng có ít nhất một giá trị numeric nằm ngoài khoảng sinh học
# Create outlier flags for each numeric column
outlier_flags = pd.DataFrame(index=df_clean.index)

for col, (low, high) in BIOLOGICAL_RANGES.items():
    if col in df_clean.columns:
        valid = df_clean[col].dropna()
        outlier_mask = (valid < low) | (valid > high)
        outlier_flags[f"{col}_outlier"] = False
        outlier_flags.loc[valid.index, f"{col}_outlier"] = outlier_mask

# Count outliers per row
outlier_count = outlier_flags.sum(axis=1)
df_clean["outlier_flag"] = (outlier_count > 0).astype(int)

print(f"Rows with at least one outlier: {df_clean['outlier_flag'].sum()} ({df_clean['outlier_flag'].sum()/len(df_clean)*100:.1f}%)")
print(f"Rows without outliers: {(df_clean['outlier_flag']==0).sum()} ({(df_clean['outlier_flag']==0).sum()/len(df_clean)*100:.1f}%)")

# Show top columns by outlier count
outlier_summary = {}
for col in outlier_flags.columns:
    cnt = outlier_flags[col].sum()
    outlier_summary[col] = cnt

print("\n=== Top 10 Columns by Outlier Count ===")
for col, cnt in sorted(outlier_summary.items(), key=lambda x: -x[1])[:10]:
    print(f"  {col}: {cnt} ({cnt/len(df_clean)*100:.1f}%)")

In [ ]:
print("=== FINAL DATA SUMMARY ===")
print(f"Shape: {df_clean.shape}")
print(f"\nData types:")
print(df_clean.dtypes)

print(f"\n=== Missing Values ===")
missing_final = df_clean.isnull().sum()
missing_pct_final = (missing_final / len(df_clean) * 100).round(2)
missing_final_df = pd.DataFrame({"Missing": missing_final, "Pct": missing_pct_final})
print(missing_final_df[missing_final_df["Missing"] > 0].sort_values("Pct", ascending=False))

print(f"\n=== Duplicates ===")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")

print(f"\n=== Outlier Summary ===")
print(f"Rows with outliers: {df_clean['outlier_flag'].sum()} ({df_clean['outlier_flag'].sum()/len(df_clean)*100:.1f}%)")

In [ ]:
# Export cleaned data
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Cleaned data exported to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

- **Input:** `Dataset/Data/CBC.csv` - 9,165 rows × 21 cols.
- **Output:** `Dataset/Clean_Data/CBC_clean.csv` - 8,147 rows × 22 cols (sau khi xử lý artifact, missing, duplicates, outliers).
- **Artifact:** Đã thay thế 4,284 giá trị `5.3976e-79` bằng `NaN`.
- **Missing:** Đã xóa 1,018 dòng thiếu hoàn toàn (11.1%). Tỷ lệ missing còn lại theo cột được báo cáo.
- **Duplicates:** Không có.
- **Outliers:** Thêm cột `outlier_flag` đánh dấu 7,860 dòng (96.5%) có ≥1 giá trị ngoài range sinh lý.
- **Labels:** Dataset vẫn không có cột target; cần merge với dataset khác để có nhãn bệnh.

| Chỉ tiêu | Trước | Sau |
|----------|-------|-----|
| Số dòng | 9,165 | 8,147 |
| Số cột | 21 | 21 + 1 (`outlier_flag`) |
| Artifact | 4,284 giá trị | 0 |
| Dòng all-NaN | 1,018 (11.1%) | 0 |
| Duplicates | 0 | 0 |
| Outlier flag | - | Có (96.5% dòng có ≥1 outlier) |
| Target | Không có | Không có (cần merge)